# Day 048 — Exercise 4: train_model + evaluate_model

**What you'll build:**
- `train_model(X_train, y_train) -> LinearRegression` — fit a linear model
- `evaluate_model(model, X_test, y_test) -> dict` — compute R² and RMSE

**Why it matters:** R² (coefficient of determination) tells you what fraction of variance in y the model explains — 1.0 is perfect, 0.0 is as good as predicting the mean, negative means worse than the mean. RMSE (root mean squared error) tells you the typical prediction error in the same units as y, so '$9,500 RMSE on house prices' is immediately interpretable.

## Provided: Setup + all prior functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }


def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded


from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """Fit a StandardScaler on training data only."""
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """Transform X using a fitted scaler; return DataFrame with same columns."""
    scaled = scaler.transform(X)
    return pd.DataFrame(scaled, columns=X.columns, index=X.index)

## Your Implementation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


def train_model(X_train: pd.DataFrame,
                y_train: pd.Series) -> LinearRegression:
    """
    Fit a LinearRegression model on training data.
    Returns the fitted model.
    """
    # TODO: model = LinearRegression()
    # TODO: model.fit(X_train, y_train)
    # TODO: return model
    pass


def evaluate_model(model: LinearRegression,
                   X_test: pd.DataFrame,
                   y_test: pd.Series) -> dict:
    """
    Evaluate a fitted model on test data.

    Returns dict with keys:
        r2          — coefficient of determination (higher = better)
        rmse        — root mean squared error (lower = better)
        n_test      — number of test samples
        predictions — numpy array of predicted values
    """
    # TODO: y_pred = model.predict(X_test)
    # TODO: r2   = r2_score(y_test, y_pred)
    # TODO: rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    # TODO: return {
    #     'r2':          round(float(r2), 4),
    #     'rmse':        round(rmse, 2),
    #     'n_test':      len(y_test),
    #     'predictions': y_pred,
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Use clean synthetic data for deterministic R²
    rng    = np.random.default_rng(0)
    n      = 200
    X_clean = pd.DataFrame({
        'a': rng.uniform(0, 10, n),
        'b': rng.uniform(0, 5,  n),
    })
    y_clean = pd.Series(5 * X_clean['a'] + 2 * X_clean['b']
                        + rng.standard_normal(n) * 0.01)
    split   = split_data(X_clean, y_clean, test_size=0.2, random_state=42)

    # Check 1: train_model returns fitted LinearRegression
    try:
        assert 'train_model' in globals()
        model = train_model(split['X_train'], split['y_train'])
        assert isinstance(model, LinearRegression), \
            f'expected LinearRegression, got {type(model).__name__}'
        assert hasattr(model, 'coef_'), 'model must be fitted (has coef_ attribute)'
        passed += 1; print('\u2705 Check 1: train_model returns fitted LinearRegression')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: coef_ length == n_features
    try:
        assert len(model.coef_) == split['n_features'], \
            f'coef_ length {len(model.coef_)} != n_features {split["n_features"]}'
        passed += 1; print(f'\u2705 Check 2: coef_ has {len(model.coef_)} entries (one per feature)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: evaluate_model returns dict with required keys
    try:
        assert 'evaluate_model' in globals()
        result = evaluate_model(model, split['X_test'], split['y_test'])
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        for k in ('r2', 'rmse', 'n_test', 'predictions'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 3: evaluate_model returns dict with all 4 keys')
    except Exception as e:
        print(f'\u274c Check 3: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 4: R² > 0.99 on clean linear data
    try:
        assert result['r2'] > 0.99, \
            f'R² on near-perfect linear data should be > 0.99, got {result["r2"]}'
        passed += 1; print(f'\u2705 Check 4: R²={result["r2"]} > 0.99 on clean data')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: RMSE > 0 (not identically perfect)
    try:
        assert result['rmse'] > 0, \
            f'rmse should be > 0, got {result["rmse"]}'
        passed += 1; print(f'\u2705 Check 5: rmse={result["rmse"]:.4f} > 0')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


def train_model(X_train: pd.DataFrame,
                y_train: pd.Series) -> LinearRegression:
    """Fit LinearRegression on training data."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model


def evaluate_model(model: LinearRegression,
                   X_test: pd.DataFrame,
                   y_test: pd.Series) -> dict:
    """Return R², RMSE, n_test, and predictions array."""
    y_pred = model.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    return {
        'r2':          round(float(r2), 4),
        'rmse':        round(rmse, 2),
        'n_test':      len(y_test),
        'predictions': y_pred,
    }
```

</details>